# 04 — Evaluation Harness

Evaluates the **shipped** pipeline (`src/rag_pipeline.py`) on the **shipped** index (`app/demo_index/`, the SAMPLE-watermarked policy) — the exact code and data the live app and API serve. The harness logic lives in `src/evaluate.py`; this notebook is a thin runner over it (you can also run `python -m src.evaluate` from the repo root).

**What it measures**
- *Out-of-scope* questions: the system correctly **abstains** with the exact `"I don't know"` response (the key guardrail metric).
- *In-scope* questions: retrieval returned chunks, the question was answered, and the answer contains the expected keywords.

**Cost / free-tier note** — calls are sequential and paced. In-scope questions cost 1 embedding + 1 generation each; out-of-scope questions short-circuit after retrieval (1 embedding, no generation). Results are cached to `eval/eval_results.json`, keyed on question text + index + model + threshold, so a reworded question or a different index re-runs instead of reusing a stale result.

**Prerequisites** — `GEMINI_API_KEY` set (env var, or Colab Secrets). No need to run NB01–NB03 first.

## 1. Locate the repo and import the pipeline

In [ ]:
import os, sys, json
from pathlib import Path

# Repo root: INSURANCE_RAG_REPO if set, else the parent of this notebooks/ folder.
REPO_ROOT = Path(os.environ.get('INSURANCE_RAG_REPO', Path.cwd().resolve().parent))
assert (REPO_ROOT / 'src' / 'rag_pipeline.py').exists(), (
    f'src/rag_pipeline.py not found under {REPO_ROOT}. Set INSURANCE_RAG_REPO to the repo root.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import rag_pipeline as rp
from src import evaluate as ev

# Index to evaluate. Default = the demo index the live app ships.
INDEX_DIR = Path(os.environ.get('INSURANCE_RAG_EVAL_INDEX', REPO_ROOT / 'app' / 'demo_index'))
QUESTIONS_PATH = ev.DEFAULT_QUESTIONS
RESULTS_PATH = ev.DEFAULT_RESULTS
SLEEP_BETWEEN = 2.0  # seconds between API calls (free-tier RPM)

print('Repo  :', REPO_ROOT)
print('Index :', INDEX_DIR)
print('Model :', rp.GEN_MODEL, '| threshold', rp.DISTANCE_THRESHOLD, '| k', rp.K_DEFAULT)

## 2. Load the question set and the index
Sanity checks run before any quota is spent.

In [ ]:
QSET = json.loads(Path(QUESTIONS_PATH).read_text(encoding='utf-8'))
print(f"Loaded {len(QSET['in_scope'])} in-scope and {len(QSET['out_of_scope'])} out-of-scope questions.")

collection = rp.load_persistent_collection(persist_dir=str(INDEX_DIR))
print(f'Index opened: {collection.count()} chunks')

rp._read_api_key()  # raises now (not mid-run) if no key is configured
print('API key found. Ready to evaluate.')

## 3. Run the evaluation
Sequential + paced + cached. Set `USE_CACHE = False` to force a fresh run.

In [ ]:
USE_CACHE = True

cache = {}
if USE_CACHE and Path(RESULTS_PATH).exists():
    cache = json.loads(Path(RESULTS_PATH).read_text(encoding='utf-8'))
print(f'{len(cache)} cached result(s) found.')

def save_cache(c):
    Path(RESULTS_PATH).parent.mkdir(parents=True, exist_ok=True)
    Path(RESULTS_PATH).write_text(json.dumps(c, indent=2, ensure_ascii=False), encoding='utf-8')

try:
    index_label = str(INDEX_DIR.resolve().relative_to(REPO_ROOT))
except ValueError:
    index_label = str(INDEX_DIR.resolve())

results, summary = ev.run_eval(
    QSET,
    answer_fn=lambda q: rp.answer_question(collection, q),
    index_label=index_label,
    cache=cache,
    save_cache=save_cache,
    sleep_between=SLEEP_BETWEEN,
)

## 4. Results table

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(results)
    cols = ['id', 'category', 'n_retrieved', 'top_distance', 'abstained', 'keyword_pass', 'question']
    display(df[[c for c in cols if c in df.columns]])
except ImportError:
    for r in results:
        print(r['id'], r['category'], 'retr=', r['n_retrieved'], 'dist=', r.get('top_distance'),
              'abstain=', r['abstained'], 'kw=', r.get('keyword_pass'))

## 5. Summary metrics

In [ ]:
print(ev.format_summary(summary, index_label))

## 6. Interpreting results

- If an in-scope question wrongly abstains, check its `top_distance` against `DISTANCE_THRESHOLD` (0.37). Prefer fixing the question wording to match the policy's own terms before moving the threshold — the threshold is calibrated for the whole set.
- If off-topic chunks leak into out-of-scope questions (abstention < 100%), the threshold is too loose.
- After editing `expected_keywords` or question wording in `eval/eval_questions.json`, just re-run: the cache key includes the question text, so changed questions are re-evaluated automatically.